# Data preparation

## Processing Guerre et Paix

In [29]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "ebooklib", "beautifulsoup4", "-q"])

import re
import unicodedata
from pathlib import Path
from ebooklib import epub
import ebooklib
from bs4 import BeautifulSoup

# ─── Configuration ───────────────────────────────────────────────────────────

BASE_DIR = Path("guerre_et_paix")
EPUB_FILES = [
    "Guerre et Paix T1.epub",
    "Guerre et Paix T2.epub",
    "Guerre et Paix T3.epub",
]

# Titles matching these are metadata/boilerplate, not chapters
SKIP_EXACT = {'FIN', 'NOTES:'}
SKIP_SUBSTRINGS = [
    'GUTENBERG', 'TRADUCTION', 'AVANT TILSITT', "L'INVASION",
    'BORODINO', 'COMTE L', 'FIN DU', 'THE FULL',
]

# ─── Helpers ─────────────────────────────────────────────────────────────────

def slugify(title: str) -> str:
    """Convert a title to a safe filesystem name (strips accents, replaces spaces)."""
    title = unicodedata.normalize('NFD', title)
    title = ''.join(c for c in title if not unicodedata.combining(c))
    title = re.sub(r'[^\w\s]', '', title)
    title = re.sub(r'\s+', '_', title.strip())
    return title


def should_skip(title: str) -> bool:
    t = title.strip().upper()
    if t in SKIP_EXACT:
        return True
    return any(kw in t for kw in SKIP_SUBSTRINGS)


def get_epub_item_content(book, filename: str) -> bytes:
    """Return the raw HTML bytes for the epub item whose name ends with filename."""
    for item in book.get_items_of_type(ebooklib.ITEM_DOCUMENT):
        if item.get_name().endswith(filename) or filename in item.get_name():
            return item.get_content()
    return b''


def parse_href(href: str):
    """Split 'file.xhtml#anchor' into ('file.xhtml', 'anchor')."""
    if href and '#' in href:
        return href.rsplit('#', 1)
    return href, None


def extract_text_between_anchors(html_bytes: bytes, start_id: str, end_id: str = None) -> str:
    """
    Extract text from the element with id=start_id up to (not including)
    the element with id=end_id.
    Handles anchors that are headings (<h3 id="...">) or embedded inside
    text elements (<p><a id="...">).
    """
    soup = BeautifulSoup(html_bytes, 'html.parser')
    TEXT_TAGS = {'p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'blockquote'}

    collecting = False
    paragraphs = []
    seen = set()  # object ids of already-collected tags (avoids duplicates from nesting)

    for tag in soup.find_all(True):
        tag_id = tag.get('id', '')

        # Stop before the next chapter's anchor
        if collecting and end_id and tag_id == end_id:
            break

        # Start collecting when we reach the chapter anchor.
        # Also handles <p><a id="anchor"> via find() on the parent text element.
        if not collecting:
            if tag_id == start_id:
                collecting = True
            elif tag.name in TEXT_TAGS and tag.find(id=start_id):
                collecting = True

        if collecting and tag.name in TEXT_TAGS and id(tag) not in seen:
            # Skip nested text tags whose parent was already collected
            if not any(id(p) in seen for p in tag.parents if p.name in TEXT_TAGS):
                text = tag.get_text(separator=' ', strip=True)
                if text:
                    paragraphs.append(text)
                seen.add(id(tag))

    return '\n\n'.join(paragraphs)


# ─── TOC parsing ─────────────────────────────────────────────────────────────

def build_chapter_list(toc) -> list:
    """
    Walk the epub TOC and return a flat list of
        (path_parts: list[str], chapter_name: str, href: str)
    where path_parts encodes the hierarchy TOME > PARTIE > CHAPITRE.
    """
    chapters = []
    state = {
        'tome': None,    'tome_i': 0,
        'partie': None,  'partie_i': 0,
        'chapitre': None,'chapitre_i': 0,
        'ch_i': 0,
    }

    def process(items):
        for item in items:
            if isinstance(item, tuple):
                section, children = item
                title = (section.title or '').strip()
                t = title.upper()

                if 'TOME' in t:
                    state['tome_i'] += 1
                    state['partie_i'] = state['chapitre_i'] = 0
                    state['tome'] = f"{state['tome_i']:02d}_{slugify(title)}"
                    state['partie'] = None
                    # Children of TOME are metadata links — do not recurse

                elif 'PARTIE' in t:
                    state['partie_i'] += 1
                    state['chapitre_i'] = 0
                    state['partie'] = f"{state['partie_i']:02d}_{slugify(title)}"
                    # Children of PARTIE are subtitle links — do not recurse

                elif 'CHAPITRE' in t or 'CHAPTER' in t:
                    state['chapitre_i'] += 1
                    state['ch_i'] = 0
                    state['chapitre'] = f"{state['chapitre_i']:02d}_{slugify(title)}"
                    process(children)  # children are the actual chapter links

                else:
                    process(children)

            elif isinstance(item, epub.Link):
                title = (item.title or '').strip()
                if should_skip(title) or not item.href or not state['chapitre']:
                    continue
                state['ch_i'] += 1
                ch_name = f"{state['ch_i']:02d}_{slugify(title)}"
                path = [p for p in [state['tome'], state['partie'], state['chapitre']] if p]
                chapters.append((path, ch_name, item.href))

    process(toc)
    return chapters


# ─── Main processing ─────────────────────────────────────────────────────────

for epub_file in EPUB_FILES:
    epub_path = BASE_DIR / epub_file
    book_name = epub_file[:-5]   # strip .epub
    book_dir  = BASE_DIR / book_name

    print(f"\n{'='*60}")
    print(f"Processing: {epub_file}")

    book = epub.read_epub(str(epub_path))
    html_cache: dict = {}

    def get_html(filename: str) -> bytes:
        if filename not in html_cache:
            html_cache[filename] = get_epub_item_content(book, filename)
        return html_cache[filename]

    chapters = build_chapter_list(book.toc)
    print(f"Chapters found: {len(chapters)}")

    for i, (path_parts, ch_name, href) in enumerate(chapters):
        next_href = chapters[i + 1][2] if i + 1 < len(chapters) else None

        curr_file, curr_anchor = parse_href(href)
        next_file, next_anchor = parse_href(next_href) if next_href else (None, None)

        # Build output directory: book_dir / TOME / PARTIE / CHAPITRE / chapter_name
        chapter_dir = book_dir
        for part in path_parts:
            chapter_dir = chapter_dir / part
        chapter_dir = chapter_dir / ch_name
        chapter_dir.mkdir(parents=True, exist_ok=True)

        # Use next_anchor as a boundary only when both chapters share the same HTML file
        end_anchor = next_anchor if (next_file == curr_file) else None

        text = ''
        if curr_anchor:
            text = extract_text_between_anchors(get_html(curr_file), curr_anchor, end_anchor)

        txt_path = chapter_dir / f"{ch_name}.txt"
        txt_path.write_text(text, encoding='utf-8')

    print(f"Saved to: {book_dir}/")

print("\nDone!")



Processing: Guerre et Paix T1.epub
Chapters found: 104
Saved to: guerre_et_paix\Guerre et Paix T1/

Processing: Guerre et Paix T2.epub
Chapters found: 102
Saved to: guerre_et_paix\Guerre et Paix T2/

Processing: Guerre et Paix T3.epub
Chapters found: 134
Saved to: guerre_et_paix\Guerre et Paix T3/

Done!


## Processing Les Miserables

In [30]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "ebooklib", "beautifulsoup4", "-q"])

import re
import unicodedata
from pathlib import Path
from ebooklib import epub
import ebooklib
from bs4 import BeautifulSoup

# ─── Configuration ───────────────────────────────────────────────────────────

BASE_DIR = Path("les_miserables")
EPUB_FILES = sorted(f.name for f in BASE_DIR.glob("*.epub"))

# Top-level links to skip (not chapters)
SKIP_TITLES = {'Page titre', '\u00c0 propos de cette \u00e9dition \u00e9lectronique'}

# ─── Helpers ─────────────────────────────────────────────────────────────────

def slugify_lm(title: str, max_len: int = 40) -> str:
    """Convert a title to a safe filesystem name, truncated to max_len.
    max_len=40 keeps total paths under Windows MAX_PATH (260 chars).
    """
    title = unicodedata.normalize('NFD', str(title))
    title = ''.join(c for c in title if not unicodedata.combining(c))
    title = re.sub(r'[^\w\s]', '', title)
    title = re.sub(r'\s+', '_', title.strip())
    return title[:max_len]


def get_epub_item_lm(book, filename: str) -> bytes:
    """Return the raw bytes of an epub item by filename."""
    for item in book.get_items_of_type(ebooklib.ITEM_DOCUMENT):
        name = item.get_name()
        if name.endswith(filename) or name.endswith('/' + filename):
            return item.get_content()
    return b''


def extract_all_text_lm(html_bytes: bytes) -> str:
    """Extract all paragraph/heading text from an HTML file (no anchor slicing needed)."""
    soup = BeautifulSoup(html_bytes, 'html.parser')
    TEXT_TAGS = {'p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'blockquote'}
    seen = set()
    paragraphs = []

    for tag in soup.find_all(TEXT_TAGS):
        if id(tag) in seen:
            continue
        if any(id(p) in seen for p in tag.parents if p.name in TEXT_TAGS):
            continue
        text = tag.get_text(separator=' ', strip=True)
        if text:
            paragraphs.append(text)
        seen.add(id(tag))

    return '\n\n'.join(paragraphs)


# ─── TOC parsing ─────────────────────────────────────────────────────────────

def build_chapter_list_lm(toc) -> list:
    """
    Parse the Les Miserables epub TOC and return a flat list of
        (livre_folder: str, chapitre_folder: str, href: str).
    Structure: LIVRE (section) -> CHAPITRE (link with full title).
    """
    chapters = []
    livre_i = 0

    for item in toc:
        if isinstance(item, tuple):
            section, children = item
            livre_title = (section.title or '').strip()
            livre_i += 1
            ch_i = 0
            livre_folder = f"{livre_i:02d}_{slugify_lm(livre_title)}"

            for child in children:
                if isinstance(child, epub.Link):
                    ch_title = (child.title or '').strip()
                    if ch_title in SKIP_TITLES or not child.href:
                        continue
                    ch_i += 1
                    ch_folder = f"{ch_i:02d}_{slugify_lm(ch_title)}"
                    chapters.append((livre_folder, ch_folder, child.href))

        elif isinstance(item, epub.Link):
            # Standalone top-level links (Page titre, A propos...) -- skip
            pass

    return chapters


# ─── Main processing ─────────────────────────────────────────────────────────

for epub_file in EPUB_FILES:
    epub_path = BASE_DIR / epub_file
    book_name = epub_file[:-5]   # strip .epub
    book_dir  = BASE_DIR / book_name

    print(f"\n{'='*60}")
    print(f"Processing: {epub_file}")

    book = epub.read_epub(str(epub_path))
    html_cache_lm: dict = {}

    def get_html_lm(filename: str) -> bytes:
        if filename not in html_cache_lm:
            html_cache_lm[filename] = get_epub_item_lm(book, filename)
        return html_cache_lm[filename]

    chapters = build_chapter_list_lm(book.toc)
    print(f"Chapters found: {len(chapters)}")

    for livre_folder, ch_folder, href in chapters:
        chapter_dir = book_dir / livre_folder / ch_folder
        chapter_dir.mkdir(parents=True, exist_ok=True)

        text = extract_all_text_lm(get_html_lm(href))

        txt_path = chapter_dir / f"{ch_folder}.txt"
        txt_path.write_text(text, encoding='utf-8')

    print(f"Saved to: {book_dir}/")

print("\nDone!")



Processing: hugo_les_miserables_cosette.epub
Chapters found: 76
Saved to: les_miserables\hugo_les_miserables_cosette/

Processing: hugo_les_miserables_fantine.epub
Chapters found: 70
Saved to: les_miserables\hugo_les_miserables_fantine/

Processing: hugo_les_miserables_idylle_plumet_epopee_st_denis.epub
Chapters found: 76
Saved to: les_miserables\hugo_les_miserables_idylle_plumet_epopee_st_denis/

Processing: hugo_les_miserables_jean_valjean.epub
Chapters found: 67
Saved to: les_miserables\hugo_les_miserables_jean_valjean/

Processing: hugo_les_miserables_marius.epub
Chapters found: 76
Saved to: les_miserables\hugo_les_miserables_marius/

Done!


## Setting up the Propp pipeline

In [1]:
def process_book(book_path, spacy_model, mentions_detection_model, coreference_resolution_model):

    import os
    import torch
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ######################################################################################

    from propp_fr import load_text_file
    text_content = load_text_file(book_path)

    ######################################################################################

    from propp_fr import generate_tokens_df
    tokens_df = generate_tokens_df(text_content, spacy_model)

    ######################################################################################

    from propp_fr import load_tokenizer_and_embedding_model, get_embedding_tensor_from_tokens_df

    # Load the tokenizer and pre-trained embedding model
    tokenizer, embedding_model = load_tokenizer_and_embedding_model(
        mentions_detection_model["base_model_name"],
      )

    # Generate embeddings for all tokens
    tokens_embedding_tensor = get_embedding_tensor_from_tokens_df(
        text_content,
        tokens_df,
        tokenizer,
        embedding_model,
        device=device,
      )

    # Save the embedding tensor alongside other output files
    _book_stem = os.path.splitext(os.path.basename(book_path))[0]
    _tensor_path = os.path.join(os.path.dirname(book_path), f"{_book_stem}_tokens_embedding_tensor")
    torch.save(tokens_embedding_tensor, _tensor_path)

    ######################################################################################

    from propp_fr import generate_entities_df

    entities_df = generate_entities_df(
        tokens_df,
        tokens_embedding_tensor,
        mentions_detection_model,
    )

    ######################################################################################

    from propp_fr import add_features_to_entities

    entities_df = add_features_to_entities(entities_df, tokens_df)

    ######################################################################################

    from propp_fr import perform_coreference

    entities_df = perform_coreference(
        entities_df,
        tokens_embedding_tensor,
        coreference_resolution_model,
        )

    ######################################################################################

    from propp_fr import extract_attributes
    tokens_df = extract_attributes(entities_df, tokens_df)

    ######################################################################################

    from propp_fr import generate_characters_dict

    characters_dict = generate_characters_dict(tokens_df, entities_df)

    ######################################################################################

    return tokens_df, entities_df, characters_dict

### Running Propp algorithm

In [3]:
import os
import gc
import datetime
import traceback
import torch
from propp_fr import load_models, save_tokens_df, save_entities_df, save_book_file

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Загружаем модели один раз для всех книг.
# Если GPU недоступен, propp_fr загружает модели с pickle без map_location='cpu',
# поэтому временно подменяем torch.load чтобы принудительно использовать CPU.
if not torch.cuda.is_available():
    _original_torch_load = torch.load
    def _cpu_torch_load(f, map_location=None, **kwargs):
        return _original_torch_load(f, map_location=map_location or 'cpu', **kwargs)
    torch.load = _cpu_torch_load
    try:
        spacy_model, mentions_detection_model, coreference_resolution_model = load_models()
    finally:
        torch.load = _original_torch_load
else:
    spacy_model, mentions_detection_model, coreference_resolution_model = load_models()

for subdir, dirs, files in os.walk("les_miserables"):
    for file in files:
        if not file.endswith(".txt"):
            continue

        book_name = file[:-4]
        book_path = os.path.join(subdir, file)

        # Пропускаем уже обработанные файлы
        book_file_path = os.path.join(subdir, book_name + "_book_file.book")
        if os.path.exists(book_file_path):
            print(f"Skipping (already processed): {file}")
            continue

        print("####################################################################################")
        print(f"Processing file: {file}")
        print(f"Start time: {datetime.datetime.now()}")

        try:
            tokens, entities, characters = process_book(
                book_path, spacy_model, mentions_detection_model, coreference_resolution_model
            )
            save_tokens_df(tokens, book_name + "_tokens", subdir)
            save_entities_df(entities, book_name + "_entities", subdir)
            save_book_file(characters, book_name + "_book_file", subdir)
            print(f"End time: {datetime.datetime.now()}")
        except Exception as e:
            print(f"ERROR processing {file}: {e}")
            traceback.print_exc()
        finally:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

Using device: cuda
Loading models...
CUDA is required, Spacy model should run on GPU.
Model Loaded Successfully from local path: C:\Users\covre\PycharmProjects\balzac_comedie_humaine\AntoineBourgois/propp-fr_NER_camembert-large_FAC_GPE_LOC_PER_TIME_VEH/final_model.pkl
Model Loaded Successfully from local path: C:\Users\covre\PycharmProjects\balzac_comedie_humaine\AntoineBourgois/propp-fr_coreference-resolution_camembert-large_PER/final_model

Models Loaded Successfully:
Spacy: fr_dep_news_trf
Mentions Detection: AntoineBourgois/propp-fr_NER_camembert-large_FAC_GPE_LOC_PER_TIME_VEH
Coreference Resolution: AntoineBourgois/propp-fr_coreference-resolution_camembert-large_PER
####################################################################################
Processing file: 01_Chapitre_I_Ce_quon_rencontre_en_venant_d.txt
Start time: 2026-04-12 19:26:46.889331


Batch Spacy Tokenization:   0%|          | 0/1 [00:00<?, ?it/s]C:\Users\covre\PycharmProjects\balzac_comedie_humaine\.venv\Lib\site-packages\thinc\util.py:401: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
Some weights of CamembertModel were not initialized from the model checkpoint at almanach/camembert-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded encoder model: almanach/camembert-large


Token indices sequence length is longer than the specified maximum sequence length for this model (945 > 512). Running this sequence through the model will result in indexing errors


Generating Mention Pairs Features Array...


...

Postprocessing Mention Pairs Predictions...


End time: 2026-04-12 19:36:00.054458
####################################################################################
Processing file: 04_Chapitre_IV_Entree_en_scene_dune_poupee.txt
Start time: 2026-04-12 19:36:00.250908


Batch Spacy Tokenization:   0%|          | 0/1 [00:00<?, ?it/s]C:\Users\covre\PycharmProjects\balzac_comedie_humaine\.venv\Lib\site-packages\thinc\util.py:401: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
Some weights of CamembertModel were not initialized from the model checkpoint at almanach/camembert-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded encoder model: almanach/camembert-large


KeyboardInterrupt: 

### Creating Guerre et Paix csv file

In [16]:
import pandas as pd
from pathlib import Path
import re

def parse_path_parts(parts):
    """Strip leading index prefix (e.g. '01_TOME_I' -> 'TOME_I') from each path part."""
    return [re.sub(r"^\d+_", "", p) for p in parts]

book_dir = Path("guerre_et_paix")

dfs = []
for book_file in sorted(book_dir.rglob("*_book_file.book")):
    # structure: guerre_et_paix / volume / tome / partie / chapitre / section / file
    rel_parts = book_file.relative_to(book_dir).parts
    volume, tome, partie, chapitre, section = rel_parts[0], *parse_path_parts(rel_parts[1:5])

    df = pd.read_json(book_file)
    df.insert(0, "volume",   volume)
    df.insert(1, "tome",     tome)
    df.insert(2, "partie",   partie)
    df.insert(3, "chapitre", chapitre)
    df.insert(4, "section",  section)
    dfs.append(df)

guerre_et_paix_df = pd.concat(dfs, ignore_index=True)
guerre_et_paix_df = guerre_et_paix_df.drop(["volume", "partie"], axis="columns")

In [17]:
guerre_et_paix_df

,tome,chapitre,section,id,count,gender,number,mentions,agent,patient,mod,poss
0,TOME_I,CHAPITRE_PREMIER,I,0,"{'occurrence': 131, 'mention_ratio': 0.3754}","{'ratio': 0.374, 'inference': {'Male': 0.88, '...","{'ratio': 0.7252000000000001, 'inference': {'S...","{'proper': [{'n': 'au prince basile', 'c': 1},...","[{'w': 'cesser', 'i': 34}, {'w': 'dire', 'i': ...","[{'w': 'voir', 'i': 268}, {'w': 'chercher', 'i...","[{'w': 'fidèle', 'i': 41}, {'w': 'cher', 'i': ...","[{'w': 'entourage', 'i': 143}, {'w': 'soirée',..."
1,TOME_I,CHAPITRE_PREMIER,I,1,"{'occurrence': 35, 'mention_ratio': 0.1003}","{'ratio': 0.4857, 'inference': {'Male': 0.24, ...","{'ratio': 0.8857, 'inference': {'Singular': 1....","{'proper': [{'n': 'anna pavlovna schérer', 'c'...","[{'w': 'dire', 'i': 9}, {'w': 'déclarer', 'i':...","[{'w': 'trouver', 'i': 1598}, {'w': 'avoir', '...","[{'w': 'sûr', 'i': 86}, {'w': 'pauvre', 'i': 2...","[{'w': 'ami', 'i': 38}, {'w': 'esclave', 'i': ..."
2,TOME_I,CHAPITRE_PREMIER,I,2,"{'occurrence': 33, 'mention_ratio': 0.0946}","{'ratio': 0.2121, 'inference': {'Male': 0.5700...","{'ratio': 0.7879, 'inference': {'Singular': 1....","{'proper': [{'n': 'anna pavlovna', 'c': 1}], '...","[{'w': 'croire', 'i': 897}, {'w': 'tenir', 'i'...","[{'w': 'irriter', 'i': 819}, {'w': 'prier', 'i...","[{'w': 'tout', 'i': 904}]","[{'w': 'avis', 'i': 848}, {'w': 'foi', 'i': 11..."
3,TOME_I,CHAPITRE_PREMIER,I,3,"{'occurrence': 15, 'mention_ratio': 0.04300000...","{'ratio': 0.2, 'inference': {'Male': 1.0, 'Fem...","{'ratio': 0.8667, 'inference': {'Singular': 1....","{'proper': [{'n': 'le prince basile', 'c': 1},...","[{'w': 'dire', 'i': 2128}, {'w': 'indiquer', '...",[],[],"[{'w': 'conclusion', 'i': 2142}, {'w': 'empres..."
4,TOME_I,CHAPITRE_PREMIER,I,4,"{'occurrence': 12, 'mention_ratio': 0.0344}","{'ratio': 0.5, 'inference': {'Male': 0.0, 'Fem...","{'ratio': 1.0, 'inference': {'Singular': 1.0, ...","{'proper': [{'n': 'anna pavlovna', 'c': 1}], '...","[{'w': 'continuer', 'i': 1638}, {'w': 'rapproc...",[],[],"[{'w': 'sourire', 'i': 1755}, {'w': 'bosse', '..."
...,...,...,...,...,...,...,...,...,...,...,...,...
7561,TOME_III,CHAPITRE_VI,XVI,11,"{'occurrence': 2, 'mention_ratio': 0.0125}","{'ratio': 0, 'inference': {'Male': 0, 'Female'...","{'ratio': 1.0, 'inference': {'Singular': 0.0, ...","{'proper': [], 'common': [{'n': 'les rostow', ...","[{'w': 'être', 'i': 116}]","[{'w': 'rencontrer', 'i': 671}]",[],[]
7562,TOME_III,CHAPITRE_VI,XVI,12,"{'occurrence': 2, 'mention_ratio': 0.0125}","{'ratio': 1.0, 'inference': {'Male': 1.0, 'Fem...","{'ratio': 1.0, 'inference': {'Singular': 1.0, ...","{'proper': [{'n': 'pierre', 'c': 1}], 'common'...","[{'w': 'supposer', 'i': 492}, {'w': 'savoir', ...",[],[],[]
7563,TOME_III,CHAPITRE_VI,XVI,13,"{'occurrence': 2, 'mention_ratio': 0.0125}","{'ratio': 0.5, 'inference': {'Male': 0.0, 'Fem...","{'ratio': 0.5, 'inference': {'Singular': 0.0, ...","{'proper': [], 'common': [{'n': 'ces demoisell...",[],[],[],[]
7564,TOME_III,CHAPITRE_VI,XVI,14,"{'occurrence': 2, 'mention_ratio': 0.0125}","{'ratio': 0.5, 'inference': {'Male': 1.0, 'Fem...","{'ratio': 0.5, 'inference': {'Singular': 1.0, ...","{'proper': [{'n': 'pierre', 'c': 1}], 'common'...",[],[],[],"[{'w': 'délivrance', 'i': 603}]"


In [11]:
guerre_et_paix_df.to_csv("./data/guerre_et_paix.csv")

In [12]:
# structure: les_miserables / volume / livre / chapitre / file
book_dir = Path("les_miserables")

dfs = []
for book_file in sorted(book_dir.rglob("*_book_file.book")):
    rel_parts = book_file.relative_to(book_dir).parts
    volume, livre, chapitre = rel_parts[0], *parse_path_parts(rel_parts[1:3])

    df = pd.read_json(book_file)
    df.insert(0, "volume",   volume)
    df.insert(1, "livre",    livre)
    df.insert(2, "chapitre", chapitre)
    dfs.append(df)

les_miserables_df = pd.concat(dfs, ignore_index=True)
print(f"Shape: {les_miserables_df.shape}")
print(f"Volumes: {les_miserables_df['volume'].unique().tolist()}")
print(les_miserables_df[["volume", "livre", "chapitre", "id"]].head(6).to_string())

ValueError: No objects to concatenate